In [1]:
import math
import csv

In [2]:
class NeuralNetwork:
#     Необходимо помнить, что hiddenWeights должен быть передан
#     [
#         [w1, w4, w7]
#         [w2, w5, w8]
#         [w3, w6, w9]
#     ]
#
    def __init__(self, activationFunc, deactivationFunc, hiddenWeights: list, outputWeights: list, requestedEra: int):
        self.requestedEra = requestedEra
        # Веса скрытых синапсов
        self.hiddenWeights = hiddenWeights
        # Веса выходных синапсов
        self.outputWeights = outputWeights
        self.activationFunc = activationFunc
        self.deactivationFunc = deactivationFunc


        self.MseNumerator = 0.0
        self.CurrentMSE = 0.0


        self.E = 0.1     # Скорость обучения
        self.a = 0.1     # Момент

        self.dataset = []
        self.answersList = []

        # Выходные значения скрытых нейронов
        self.hiddenNeuronOutput = []

        self.hiddenWeightDeltas = []      # Дельты скрытых синапсов
        self.outputWeightDeltas = []      # Дельты выходных синапсов

    def readFromCsv(self, datasetPath: str):
        # Чтение датасета из csv
        with open(datasetPath, 'r') as csvDataset:
            csvDatasetReader = csv.reader(csvDataset, delimiter='\t')
            for row in csvDatasetReader:
                self.dataset.append([ float(row[0]), float(row[1]), float(row[2]), float(row[3]) ])

#     def __normalizeTrainSet(self):
#         a = []
#         b = []
#         c = []
        
#         for fullTrainSet in self.dataset:
#             a.append(fullTrainSet[0])
#             b.append(fullTrainSet[1])
#             c.append(fullTrainSet[2])
        
#         aMin = min(a)
#         aMax = max(a)
#         print(f'aMin = {aMin}; aMax = {aMax}')
        
#         bMin = min(b)
#         bMax = max(b)
#         print(f'bMin = {bMin}; bMax = {bMax}')
        
#         cMin = min(c)
#         cMax = max(c)
#         print(f'cMin = {cMin}; cMax = {cMax}')
        
#         for i in range(len(self.dataset)):
            
            
        
    def train(self):
        for fullTrainSet in self.dataset:
            print('------------------------------')
            ideal = fullTrainSet[-1]
#             print(f'preactivatedIdeal = {ideal}')
#             ideal = self.activationFunc(ideal)
            trainSet = fullTrainSet[0:-1]
            print(f'trainSet = {trainSet}; ideal = {ideal}')

            hiddenNeuronInputsValues = self.__calculateHiddenInputsValues(trainSet)
            print(f'hiddenNeuronInputsValues = {hiddenNeuronInputsValues}')

            hiddenNeuronOutputsValues = self.__calculateHiddenOutputsValues(hiddenNeuronInputsValues)
            print(f'hiddenNeuronOutputsValues = {hiddenNeuronOutputsValues}')

            
            output = self.__calculateOutput(hiddenNeuronOutputsValues)
            print(f'output = {output}')

            # Запись ответа для расчёта MSE
            self.answersList.append(output)
            self.CurrentMSE = self.__calculateMSE(ideal, output)
            print(f'MSE = {self.CurrentMSE}')

            dO = (ideal - output)*self.deactivationFunc(output)
#             dO = (ideal - output)
            print(f'dO = {dO}')

            dH = []
            for i in range(len(self.outputWeights)):
                dH.append( self.deactivationFunc(hiddenNeuronOutputsValues[i])*dO*self.outputWeights[i] )
            print(f'dH = {dH}')

            print(f'previousOutputDeltas = {self.outputWeightDeltas}')
            self.outputWeightDeltas = self.__calculateOutputWeightDeltas(dO, hiddenNeuronOutputsValues)
            print(f'outputDeltas = {self.outputWeightDeltas}')
            
            for deltas in self.hiddenWeightDeltas:
                print(f'previousHiddenDeltas = {deltas}')
            self.hiddenWeightDeltas = self.__calculateHiddenWeightDeltas(trainSet, dH)
            for deltas in self.hiddenWeightDeltas:
                print(f'hiddenDeltas = {deltas}')

            self.hiddenWeights = self.__calculateNewHiddenWeigths(self.hiddenWeights, self.hiddenWeightDeltas)
            for deltas in self.hiddenWeights:
                print(f'hiddenWeights = {deltas}')

            self.outputWeights = self.__calculateNewOutputWeigths(self.outputWeights, self.outputWeightDeltas)
            print(f'outputWeights = {self.outputWeights}')
            print('------------------------------')


    def __calculateMSE(self, ideal:float, real: float):
        self.MseNumerator += ((ideal - real)**2)
        return self.MseNumerator / len(self.answersList)

    # Приходит [I_1, I_2, I_3]
    # Возвращается [H1_in, H2_in, H3_in]
    def __calculateHiddenInputsValues(self, trainSet: list) -> list:
        result = []
        for weights in self.hiddenWeights:
            halfResult = float(0)
            for i in range(len(weights)):
                halfResult += trainSet[i]*weights[i]

            result.append(halfResult)

        return result

    # Приходит [H1_in, H2_in, H3_in]
    # Возвращается [H1_out, H2_out, H3_out]
    def __calculateHiddenOutputsValues(self, inputs: list) -> list:
        result = []

        for inputValue in inputs:
            result.append(self.activationFunc(inputValue))

        return result

    # Приходит [H1_out, H2_out, H3_out]
    # Возвращается O_out
    def __calculateOutput(self, hiddenOutputsValues: list) -> float:
        result = float(0)
        for i in range(len(hiddenOutputsValues)):
            result += hiddenOutputsValues[i]*self.outputWeights[i]

        return result

    def __calculateOutputWeightDeltas(self, dO:float, hiddenNeuronOutputsValues: list) -> list:
        '''
        Расчёт дельт для весов выходных синапсов

        dO: float
        hiddenNeuronOutputsValues: list
        '''
        outputDeltas = []
        for i in range(len(hiddenNeuronOutputsValues)):
            outputGradient = dO*hiddenNeuronOutputsValues[i]

            previousOutputDelta = float(0)
            try:
                previousOutputDelta = float(self.outputWeightDeltas[i])
            except:
                pass

            delta = self.E*outputGradient + self.a*previousOutputDelta

            outputDeltas.append(delta)

        return outputDeltas


    def __calculateHiddenWeightDeltas(self, trainSet: list, dH: list) -> list:
        '''
        Расчёт дельт для весов скрытых синапсов

        trainSet: [a, b, c]
        dO: float
        output: float
        '''

        hiddenGradients = []
        for dH_n in dH:
            buffer = []
            for inputValue in trainSet:
                buffer.append( inputValue*dH_n )

            hiddenGradients.append(buffer)

        hiddenDeltas = []
        for i in range(len(hiddenGradients)):
            deltasOfLine = []
            for j in range (len(hiddenGradients[i])):
                hiddenGradient = hiddenGradients[i][j]
                previousHiddenDelta = float(0)
                try:
                    previousHiddenDelta = float(self.hiddenWeightDeltas[i])
                except:
                    pass

                delta = self.E*hiddenGradient + self.a*previousHiddenDelta

                deltasOfLine.append(delta)

            hiddenDeltas.append(deltasOfLine)

        return hiddenDeltas


    def __calculateNewHiddenWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        # Рачёт новых весов для скрытых синапсов
        for i in range(len(currentWeights)):
            lineOfNewWeights = []
            for j in range(len(currentWeights[i])):
                lineOfNewWeights.append( currentWeights[i][j] + deltas[i][j] )

            newWeights.append(lineOfNewWeights)

        return newWeights

    def __calculateNewOutputWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        # Рачёт новых весов для выходных синапсов
        for i in range(len(currentWeights)):
            newWeights.append( currentWeights[i] + deltas[i] )

        return newWeights



In [3]:
# Сигмоид
f_sig = lambda x: 1/(1 + math.exp(-1*x))
# Конечно-разностный аналог сигмоида для метода обр распределения
anti_f_sig = lambda x: (1 - x)*x

# Тангенсоид
f_tang = lambda x:(math.exp(2*x) - 1) / (math.exp(2*x) + 1)
# Конечно-разностный аналог Тангенсоид для метода обр распределения
f_anti_tang = lambda x: 1 - x**2

#RELU
relu = lambda x: max(0, x)
anti_relu = lambda x: float(x >= 0)

In [4]:
hiddenWeights = [[0.1, 0.9, 0.31], [0.4, 0.7, 0.11], [0.3, 0.2, 0.27]]
# hiddenWeights = [[1, 1, 6], [3, 7, 7], [5, 6, 0]]

outputWeights = [0.47, 0.51, 0.67]
# outputWeights = [1, 4, 6]

In [5]:
# myAI = NeuralNetwork(f_sig, anti_f_sig, hiddenWeights, outputWeights, 1000)
# myAI = NeuralNetwork(f_tang, f_anti_tang, hiddenWeights, outputWeights, 1000)
myAI = NeuralNetwork(relu, anti_relu, hiddenWeights, outputWeights, 1000)

In [6]:
myAI.readFromCsv("discriminant_data.csv")

In [7]:
myAI.train()

------------------------------
trainSet = [3.6629980747925583, 3.7093400267829733, 0.7288568205420493]; ideal = 3.079998912514501
hiddenNeuronInputsValues = [3.9306514459519675, 4.141911498924729, 2.0375587693407153]
hiddenNeuronOutputsValues = [3.9306514459519675, 4.141911498924729, 2.0375587693407153]
output = 5.3249454195073165
MSE = 5.039784819259243
dO = -2.2449465069928154
dH = [-1.0551248582866233, -1.1449227185663358, -1.5041141596851864]
previousOutputDeltas = []
outputDeltas = [-0.882410223379613, -0.9298369751784448, -0.4574210442024018]
hiddenDeltas = [-0.3864920324569672, -0.39138168700962844, -0.07690349494856685]
hiddenDeltas = [-0.419384971389475, -0.4246907667551287, -0.0834484732420619]
hiddenDeltas = [-0.5509567271195065, -0.5579270857371299, -0.10962838641604211]
hiddenWeights = [-0.2864920324569672, 0.5086183129903716, 0.23309650505143314]
hiddenWeights = [-0.01938497138947498, 0.2753092332448713, 0.026551526757938107]
hiddenWeights = [-0.2509567271195065, -0.35792

In [8]:
print("myAI.CurrentMSE", myAI.CurrentMSE)
for weight in myAI.hiddenWeights:
    print(weight)
# print("myAI.hiddenWeights", myAI.hiddenWeights)
print("myAI.outputWeights", myAI.outputWeights)
print("myAI.answersList", myAI.answersList)

myAI.CurrentMSE 513.557359058463
[2.6895190428162423, 0.5320737123001721, 0.5924386198921034]
[3.030647399399448, 0.2993480305998522, 0.3948314338988003]
[-1.194001880954425, -0.3653596859831957, 0.04650246746924637]
myAI.outputWeights [-0.5104558037551254, -0.5231521946427166, 0.16175439533066469]
myAI.answersList [5.3249454195073165, -0.472942988539045, -0.7735414712811797, -1.2334776374049228, -1.957748517577651, -0.5265530082955141, -0.7595207572674277, -1.8371330555490575, -1.347753169641737, -0.12627069681510011, -0.45182574630983735, -0.600593852589421, -2.2827890397577257, -1.9293166383241667, -1.8666574400599405, -0.4137028673562726, -0.5210977470403942, -1.5037430308714281, -1.3066676127801722, -1.2369197729041785, -0.11004294883513129, -0.47066020629618066, -0.6525789500244313, -0.3419024819836261, -0.8717933772049741, -1.301218628131505, -0.021680109170081313, -1.726075057879006, -1.3738404871954164, -1.7481659364045168, -0.9742889822386559, -0.13189310066166451, -0.0466883